In [25]:
import numpy as np
import os
from rnin_loader import *
import pandas 
import json
import h5py

def calculate_trajectory_distance(gt_data):
    """
    Calculates the total distance traveled along a 2D trajectory using numpy.

    Args:
        gt_data: A numpy array of shape (N, 2) where N is the number of points.

    Returns:
        The total distance traveled as a float.
    """
    # Calculate the distance of each step using the provided lines
    step_d = np.linalg.norm(np.diff(gt_data, axis=0), axis=1)

    # The total distance is simply the sum of all step distances
    total_distance_sequence = np.sum(step_d)

    return total_distance_sequence

def process_dataset(list_file_path, data_dir):
    """
    Reads a list of sequence filenames, processes each, and sums the distances.

    Args:
        list_file_path: The path to the text file containing the sequence names.

    Returns:
        The total distance for the entire dataset.
    """
    total_dataset_distance = 0.0
    total_frames = 0
    frequency_hz = 60

    # 1. Read the list of sequence file names from the input text file
    with open(list_file_path) as f:
        lines = [s.strip() for s in f.readlines() if len(s) > 0 and s[0] != '#']

    data_list = []
    for line in lines:
        data_name = line.split(',')[0]  # get the trajectory file name (without extension)
        data_list.append(data_name)

    distances = []
    times = []
    for seq in data_list:
        path = data_dir + seq + '_gt.npy'
        trajectory_data = np.load(path)
        num_points_in_seq = trajectory_data.shape[0]
        time_seconds = num_points_in_seq / frequency_hz
        times.append(time_seconds)
        total_frames += num_points_in_seq

        
        distance = calculate_trajectory_distance(trajectory_data)
        distances.append(distance)
        # print(f"Distance for {seq}: {distance:.2f} meters")

        # 4. Sum the distance to the total
        total_dataset_distance += distance

    total_time = total_frames / frequency_hz
    return total_dataset_distance, np.mean(distances), np.std(distances), total_time, np.mean(times), np.std(times)


def process_rnin_dataset(list_file_path, data_dir):
    """
    Reads a list of sequence filenames, processes each, and sums the distances.

    Args:
        list_file_path: The path to the text file containing the sequence names.

    Returns:
        The total distance for the entire dataset.
    """
    total_dataset_distance = 0.0
    total_frames = 0
    frequency_hz = 100

    # 1. Read the list of sequence file names from the input text file
    with open(list_file_path) as f:
        lines = [s.strip() for s in f.readlines() if len(s) > 0 and s[0] != '#']

    data_list = []
    for line in lines:
        data_name = line.split(',')[0]  # get the trajectory file name (without extension)
        data_list.append(data_name)


    distances = []
    times = []
    for seq in data_list:
        path = data_dir + seq + '/SenseINS.h5'
        imu_all = pandas.read_hdf(path, 'imu_all')
        
            
        tmp_vio_q = np.array(imu_all[['gt_q_w', 'gt_q_x', 'gt_q_y', 'gt_q_z']].values)
        if tmp_vio_q[0][0] == 1.0 and tmp_vio_q[100][0] == 1.0 or tmp_vio_q[0][0] == tmp_vio_q[-1][0]:
            tmp_vio_q = np.array(imu_all[['vio_q_w', 'vio_q_x', 'vio_q_y', 'vio_q_z']].values)
            tmp_vio_p = np.array(imu_all[['vio_p_x', 'vio_p_y']].values)
        else:
            tmp_vio_p = np.array(imu_all[['gt_p_x', 'gt_p_y']].values)
            
        trajectory_data = tmp_vio_p
        num_points_in_seq = trajectory_data.shape[0]
        time_seconds = num_points_in_seq / frequency_hz
        times.append(time_seconds)
        total_frames += num_points_in_seq

        
        distance = calculate_trajectory_distance(trajectory_data)
        distances.append(distance)
        # print(f"Distance for {seq}: {distance:.2f} meters")

        # 4. Sum the distance to the total
        total_dataset_distance += distance

    total_time = total_frames / frequency_hz
    return total_dataset_distance, np.mean(distances), np.std(distances), total_time, np.mean(times), np.std(times)


def process_ronin_dataset(list_file_path, data_dir):
    """
    Reads a list of sequence filenames, processes each, and sums the distances.

    Args:
        list_file_path: The path to the text file containing the sequence names.

    Returns:
        The total distance for the entire dataset.
    """
    total_dataset_distance = 0.0
    total_frames = 0
    frequency_hz = 200

    # 1. Read the list of sequence file names from the input text file
    with open(list_file_path) as f:
        lines = [s.strip() for s in f.readlines() if len(s) > 0 and s[0] != '#']

    data_list = []
    for line in lines:
        data_name = line.split(',')[0]  # get the trajectory file name (without extension)
        data_list.append(data_name)


    distances = []
    times = []
    for seq in data_list:
        path = data_dir + seq + '/data.hdf5'
        info_path = data_dir + seq + '/info.json'
        
        with open(info_path) as f:
            info = json.load(f)
             
        with h5py.File(path) as f:
            tango_pos = np.copy(f['pose/tango_pos'])

        start_frame = info.get('start_frame', 0)
        trajectory_data = tango_pos[start_frame:, :2]

        num_points_in_seq = trajectory_data.shape[0]
        time_seconds = num_points_in_seq / frequency_hz
        times.append(time_seconds)
        total_frames += num_points_in_seq

        
        distance = calculate_trajectory_distance(trajectory_data)
        distances.append(distance)
        # print(f"Distance for {seq}: {distance:.2f} meters")

        # 4. Sum the distance to the total
        total_dataset_distance += distance

    total_time = total_frames / frequency_hz
    return total_dataset_distance, np.mean(distances), np.std(distances), total_time, np.mean(times), np.std(times)
    
# --- Main execution block ---
if __name__ == "__main__":
    # Define the path to your file containing the list of sequence names
    file_list_name =  "./../lists/rnin/all_rnin.txt" #"./../lists/ronin/all_ronin.txt" #"./../lists/rnin/all_rnin.txt" # "./../lists/our/all_umgloc.txt"
    data_dir =  "./../data/datasets/rnin/" #"./../data/datasets/ronin/" #"./../data/datasets/rnin/" #"./../results/paper_plots/trajectory_plot_95/traj/"
    dataset = 'rnin' #'ronin' #'rnin' #'umgloc'

    # Calculate the total distance for the dataset
    if dataset == 'umgloc':
        total_distance, mean_distance, std_distance, total_time, mean_time, std_time = process_dataset(file_list_name, data_dir)

    elif dataset == 'rnin':
        total_distance, mean_distance, std_distance, total_time, mean_time, std_time = process_rnin_dataset(file_list_name, data_dir)

    elif dataset == 'ronin':
        total_distance, mean_distance, std_distance, total_time, mean_time, std_time = process_ronin_dataset(file_list_name, data_dir)

    print("-" * 30)
    print(f"Grand Total Dataset Travel Distance: {total_distance:.2f} meters")
    print(f"Dataset Mean Travel Distance: {mean_distance:.2f} meters")
    print(f"Dataset Standard Deviation Travel Distance: {std_distance:.2f} meters")
    print(f"Grand Total Dataset Travel Time: {total_time:.2f} seconds")
    print(f"Dataset Mean Travel Time: {mean_time:.2f} seconds")
    print(f"Dataset Standard Deviation Travel Time: {std_time:.2f} seconds")



------------------------------
Grand Total Dataset Travel Distance: 27818.13 meters
Dataset Mean Travel Distance: 92.42 meters
Dataset Standard Deviation Travel Distance: 160.83 meters
Grand Total Dataset Travel Time: 74139.27 seconds
Dataset Mean Travel Time: 246.31 seconds
Dataset Standard Deviation Travel Time: 318.99 seconds
